## Week 3: Forward and Backward Selection and Partial Least Squares Regression

For this week, I will be performing forward and backward selection on the datasets and then will fit the best ridge regression model for each dataset with the forward selected features and then the backward selected features. I will also perform partial least squares to see how it compares.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.cross_decomposition import PLSRegression

## Dataset 1: Instacart Dataset

In [2]:
data1 = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/instacart.csv')

transform_cols = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase']

for col in transform_cols:
    data1[f'log_{col}'] = np.log1p(data1[col])

data1_transformed = data1.drop(columns = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase'])

X = data1_transformed.drop(columns = ['ordered'])
y = data1_transformed['ordered']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

# Alpha value came from hyperparameter tuning in Week 2
model = Ridge(alpha = 0.1873817422860385)

forward = SequentialFeatureSelector(model, direction = 'forward')

forward.fit(X_train, y_train)

print(f'Selected Features: {forward.get_support()}')

Selected Features: [False False False False  True False  True  True  True]


In [4]:
X_train_forward = X_train[['log_frequency', 'log_Last Product Order', 'log_Last Order', 'log_Orders Since Last Purchase']]
X_test_forward = X_test[['log_frequency', 'log_Last Product Order', 'log_Last Order', 'log_Orders Since Last Purchase']]

model.fit(X_train_forward, y_train)

y_preds = model.predict(X_test_forward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,-0.012,0.007,0.105
2,-0.002,0.008,0.130
3,0.008,0.019,0.296
4,0.018,0.020,0.318
5,0.030,0.031,0.483
6,0.047,0.045,0.703
7,0.068,0.055,0.868
8,0.091,0.075,1.187
9,0.128,0.120,1.887


In [5]:
backward = SequentialFeatureSelector(model, direction = 'backward')

backward.fit(X_train, y_train)

print(f'Selected Features: {backward.get_support()}')

Selected Features: [False False False  True  True  True  True False  True]


In [6]:
X_train_backward = X_train[['log_num_orders', 'log_frequency', 'log_Product Popularity', 'log_Last Product Order',
                            'log_Orders Since Last Purchase']]
X_test_backward = X_test[['log_num_orders', 'log_frequency', 'log_Product Popularity', 'log_Last Product Order',
                            'log_Orders Since Last Purchase']]

model.fit(X_train_backward, y_train)

y_preds = model.predict(X_test_backward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,-0.013,0.006,0.098
2,-0.002,0.009,0.146
3,0.008,0.016,0.256
4,0.018,0.020,0.308
5,0.031,0.030,0.476
6,0.047,0.044,0.691
7,0.068,0.057,0.892
8,0.091,0.075,1.178
9,0.129,0.121,1.910


In [7]:
PLS_model = PLSRegression()

PLS_model.fit(X_train, y_train)

y_preds = PLS_model.predict(X_test)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,-0.031,0.006,0.099
2,-0.005,0.009,0.144
3,0.011,0.012,0.189
4,0.026,0.024,0.378
5,0.041,0.031,0.494
6,0.056,0.044,0.696
7,0.073,0.060,0.950
8,0.095,0.077,1.208
9,0.130,0.118,1.849


## Dataset 2: Amazon Reviews Dataset

In [8]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

small_reviews = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_reviews%5B1%5D.csv')

small_meta = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_meta%5B1%5D.csv')

small_meta = small_meta.set_index(keys = 'parent_asin')

small_meta = small_meta.drop(columns = ['title', 'images'])

data2 = small_reviews.join(small_meta, on = 'parent_asin', how = 'inner')

data2 = data2.drop(columns = ['main_category', 'store', 'rating_number', 'videos', 'images', 'subtitle', 'author', 'bought_together', 'features', 'description', 'details', 'timestamp', 'text', 'title'])

data2['product_review_count'] = data2.groupby('asin').transform('size')
data2['user_review_count'] = data2.groupby('user_id').transform('size')

data2['price'] = data2['price'].fillna(data2['price'].median())

data2['categories'] = data2['categories'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

categories = pd.DataFrame(mlb.fit_transform(data2['categories']), columns = mlb.classes_, index = data2.index)

top_30 = categories.sum().sort_values(ascending = False).head(31).index

categories = categories[top_30[1:31]]

data2 = data2.drop(columns = 'categories')

data2 = data2.join(categories)

columns_to_transform = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count']

for col in columns_to_transform:
    data2[f'log_{col}'] = np.log1p(data2[col])

data2_transformed = data2.drop(columns = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count'])

data2_transformed = data2_transformed.reset_index(drop = True)

for i in range(len(data2_transformed)):
    if data2_transformed.iloc[i]['rating'] == 4:
        data2_transformed.loc[i, 'liked'] = 1
    elif data2_transformed.iloc[i]['rating'] == 5:
        data2_transformed.loc[i, 'liked'] = 1
    else:
        data2_transformed.loc[i, 'liked'] = 0

X = data2_transformed.drop(columns = ['asin', 'parent_asin', 'user_id', 'liked', 'rating'])
y = data2_transformed['liked']

X.head()

,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,...,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
0,False,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0.000000,1.704748,3.257712,0.693147,0.693147
1,True,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,3.295837,1.686399,3.351307,0.693147,0.693147
2,True,1,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0.000000,1.686399,2.889816,0.693147,1.098612
3,True,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0.000000,1.722767,2.771964,0.693147,1.098612
4,True,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0.000000,1.686399,3.395179,1.609438,0.693147


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

# Alpha value came from hyperparameter tuning in Week 2
model = Ridge(alpha = 0.03511191734215131)

forward = SequentialFeatureSelector(model, direction = 'forward')

forward.fit(X_train, y_train)

print(f'Selected Features: {forward.get_support()}')

Selected Features: [False False False  True False False False False  True False False  True
 False  True False  True  True False  True False False False  True  True
 False  True  True  True  True  True  True False  True  True  True False]


In [10]:
X_train.columns

Index(['verified_purchase', 'Beverages', 'Snacks & Sweets', 'Pantry Staples',
       'Coffee', 'Snack Foods', 'Candy & Chocolate',
       'Bottled Beverages, Water & Drink Mixes', 'Cooking & Baking',
       'Breads & Bakery', 'Single-Serve Capsules & Pods', 'Tea', 'Cookies',
       'Meat Snacks', 'Soups, Stocks & Broths', 'Nuts & Seeds', 'Jerky',
       'Juices', 'Ground Coffee', 'Breakfast Foods', 'Cereals', 'Granola',
       'Fruit & Herbal Tea', 'Herbal', 'Bars', 'Roasted Coffee Beans',
       'Whole Coffee Beans', 'Canned, Jarred & Packaged Foods', 'Cold Cereals',
       'Baking Syrups, Sugars & Sweeteners', 'Black', 'log_helpful_vote',
       'log_average_rating', 'log_price', 'log_product_review_count',
       'log_user_review_count'],
      dtype='str')

In [11]:
X_train_forward = X_train[['Pantry Staples', 'Cooking & Baking', 'Tea', 'Meat Snacks', 'Nuts & Seeds', 'Jerky',
'Ground Coffee', 'Fruit & Herbal Tea', 'Herbal', 'Roasted Coffee Beans', 'Whole Coffee Beans',
'Canned, Jarred & Packaged Foods', 'Cold Cereals', 'Baking Syrups, Sugars & Sweeteners', 'Black', 'log_average_rating',
'log_price', 'log_product_review_count']]
X_test_forward = X_test[['Pantry Staples', 'Cooking & Baking', 'Tea', 'Meat Snacks', 'Nuts & Seeds', 'Jerky',
'Ground Coffee', 'Fruit & Herbal Tea', 'Herbal', 'Roasted Coffee Beans', 'Whole Coffee Beans',
'Canned, Jarred & Packaged Foods', 'Cold Cereals', 'Baking Syrups, Sugars & Sweeteners', 'Black', 'log_average_rating',
'log_price', 'log_product_review_count']]

model.fit(X_train_forward, y_train)

y_preds = model.predict(X_test_forward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.677,0.667,0.798
2,0.790,0.933,1.117
3,0.819,0.571,0.684
4,0.849,0.875,1.047
5,0.872,0.857,1.026
6,0.887,0.857,1.026
7,0.901,0.929,1.111
8,0.928,0.944,1.130
9,0.967,0.818,0.979


In [12]:
backward = SequentialFeatureSelector(model, direction = 'backward')

backward.fit(X_train, y_train)

print(f'Selected Features: {backward.get_support()}')

Selected Features: [False  True False False False  True False  True  True  True  True False
  True  True False  True  True False  True False False False  True  True
 False  True  True False False  True False False  True  True False False]


In [13]:
X_train_backward = X_train[['Beverages', 'Snack Foods', 'Bottled Beverages, Water & Drink Mixes', 'Cooking & Baking',
                            'Breads & Bakery', 'Single-Serve Capsules & Pods', 'Cookies', 'Meat Snacks', 'Nuts & Seeds',
                            'Jerky', 'Ground Coffee', 'Fruit & Herbal Tea', 'Herbal', 'Roasted Coffee Beans',
                            'Whole Coffee Beans', 'Baking Syrups, Sugars & Sweeteners', 'log_average_rating', 'log_price']]

X_test_backward = X_test[['Beverages', 'Snack Foods', 'Bottled Beverages, Water & Drink Mixes', 'Cooking & Baking',
                            'Breads & Bakery', 'Single-Serve Capsules & Pods', 'Cookies', 'Meat Snacks', 'Nuts & Seeds',
                            'Jerky', 'Ground Coffee', 'Fruit & Herbal Tea', 'Herbal', 'Roasted Coffee Beans',
                            'Whole Coffee Beans', 'Baking Syrups, Sugars & Sweeteners', 'log_average_rating', 'log_price']]

model.fit(X_train_backward, y_train)

y_preds = model.predict(X_test_backward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.717,0.688,0.823
2,0.784,0.857,1.026
3,0.812,0.714,0.855
4,0.838,0.733,0.878
5,0.856,1.000,1.197
6,0.880,0.769,0.921
7,0.905,1.000,1.197
8,0.937,0.850,1.017
9,0.972,0.889,1.064


In [14]:
PLS_model = PLSRegression()

PLS_model.fit(X_train, y_train)

y_preds = PLS_model.predict(X_test)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.674,0.600,0.718
2,0.775,0.733,0.878
3,0.814,0.857,1.026
4,0.839,0.867,1.037
5,0.862,0.857,1.026
6,0.884,0.867,1.037
7,0.906,0.929,1.111
8,0.937,0.800,0.957
9,0.967,0.929,1.111


## Dataset 3: Ecommerce Dataset

In [15]:
data3 = pd.read_csv('/workspaces/DX799---HW-Assignments/data.csv', encoding = 'windows-1252')
data3['Purchased'] = 1

customer_id = data3['Customer ID'].values
segments = data3['Segment'].values
cities = data3['City'].values
state = data3['State'].values
region = data3['Region'].values
product_id = data3['Product ID'].values
categories = data3['Category'].values
subcategories = data3['Sub-Category'].values

def generate_row(df):
    row = {}

    row['Customer ID'] = np.random.choice(customer_id)
    cust_id = row['Customer ID']
    purchased = set(df[df['Customer ID'] == cust_id]['Product ID'])
    candidates = list(set(product_id) - purchased)

    row['Segment'] = np.random.choice(segments)
    row['State'] = np.random.choice(state)
    row['City'] = np.random.choice(cities)
    row['Region'] = np.random.choice(region)
    row['Product ID'] = np.random.choice(candidates)
    row['Category'] = np.random.choice(categories)
    row['Sub-Category'] = np.random.choice(subcategories)

    row['Sales'] = 0
    row['Discount'] = 0
    row['Quantity'] = 0
    row['Profit'] = 0
    row['Purchased'] = 0

    return row

fake_rows = pd.DataFrame([generate_row(data3) for n in range(1000)])

fake_df = pd.concat([data3, fake_rows], ignore_index = True)

fake_df = fake_df.drop(columns = ['Row ID', 'Product Name', 'Country', 'Postal Code', 'Order ID', 'Order Date'])

fake_df_encoded = pd.get_dummies(fake_df, columns = ['Ship Mode', 'Segment', 'State', 'Region', 'Category', 'Sub-Category'], dtype = int)

X = fake_df_encoded.drop(columns = ['Customer ID', 'Product ID', 'City', 'Purchased'])
y = fake_df_encoded['Purchased']

X.head()

,Sales,Quantity,Discount,Profit,Ship Mode_First Class,Ship Mode_Same Day,Ship Mode_Second Class,Ship Mode_Standard Class,Segment_Consumer,Segment_Corporate,...,Sub-Category_Envelopes,Sub-Category_Fasteners,Sub-Category_Furnishings,Sub-Category_Labels,Sub-Category_Machines,Sub-Category_Paper,Sub-Category_Phones,Sub-Category_Storage,Sub-Category_Supplies,Sub-Category_Tables
0,48.896,4,0.2,8.5568,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
1,474.430,11,0.0,199.2606,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
2,3.600,2,0.0,1.7280,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,454.560,5,0.2,-107.9580,0,0,0,1,1,0,...,0,0,0,0,0,0,0,1,0,0
4,141.420,5,0.6,-187.3815,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0


In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

# Alpha value came from hyperparameter tuning in Week 2
model = Ridge(alpha = 0.0001)

forward = SequentialFeatureSelector(model, direction = 'forward')

forward.fit(X_train, y_train)

print(f'Selected Features: {forward.get_support()}')

Selected Features: [ True  True False  True  True  True  True  True  True False False False
  True  True  True False False False  True  True  True  True False  True
  True  True False  True  True  True  True  True  True  True  True  True
  True False  True False  True False  True False False False False  True
 False False  True False False  True False False False False False  True
  True False False  True False False False  True False  True False False
 False False  True False False False False  True False False]


In [17]:
X_train_forward = X_train[['Sales', 'Quantity', 'Ship Mode_First Class', 'Ship Mode_Same Day', 'Ship Mode_Second Class',
                           'Ship Mode_Standard Class', 'Segment_Home Office', 'State_Alabama', 'State_Arkansas',
                           'State_California', 'State_Florida', 'State_Georgia', 'State_Idaho', 'State_Illinois',
                           'State_Indiana', 'State_Kansas', 'State_Kentucky', 'State_Louisiana', 'State_Maryland',
                           'State_Michigan', 'State_Minnesota', 'State_Missouri', 'State_Montana', 'State_Nevada',
                            'State_New York', 'State_North Dakota', 'State_Pennsylvania', 'State_Tennessee',
                            'State_Vermont', 'State_Virginia', 'State_Washington', 'State_Wisconsin', 'Region_East',
                            'Sub-Category_Appliances', 'Sub-Category_Art', 'Sub-Category_Bookcases', 'Sub-Category_Copiers',
                            'Sub-Category_Furnishings', 'Sub-Category_Machines', 'Sub-Category_Supplies', 'Sub-Category_Tables']]
X_test_forward = X_test[['Sales', 'Quantity', 'Ship Mode_First Class', 'Ship Mode_Same Day', 'Ship Mode_Second Class',
                           'Ship Mode_Standard Class', 'Segment_Home Office', 'State_Alabama', 'State_Arkansas',
                           'State_California', 'State_Florida', 'State_Georgia', 'State_Idaho', 'State_Illinois',
                           'State_Indiana', 'State_Kansas', 'State_Kentucky', 'State_Louisiana', 'State_Maryland',
                           'State_Michigan', 'State_Minnesota', 'State_Missouri', 'State_Montana', 'State_Nevada',
                            'State_New York', 'State_North Dakota', 'State_Pennsylvania', 'State_Tennessee',
                            'State_Vermont', 'State_Virginia', 'State_Washington', 'State_Wisconsin', 'Region_East',
                            'Sub-Category_Appliances', 'Sub-Category_Art', 'Sub-Category_Bookcases', 'Sub-Category_Copiers',
                            'Sub-Category_Furnishings', 'Sub-Category_Machines', 'Sub-Category_Supplies', 'Sub-Category_Tables']]

model.fit(X_train_forward, y_train)

y_preds = model.predict(X_test_forward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.000,0.000,0.000
2,0.000,0.000,0.000
3,0.843,0.843,1.075
4,1.000,1.000,1.276
5,1.000,1.000,1.276
6,1.000,1.000,1.276
7,1.000,1.000,1.276
8,1.000,1.000,1.276
9,1.000,1.000,1.276


In [18]:
backward = SequentialFeatureSelector(model, direction = 'backward')

backward.fit(X_train, y_train)

print(f'Selected Features: {backward.get_support()}')

Selected Features: [False False False False  True  True  True  True False False False False
 False  True False False False False  True False False  True False  True
  True False False False False False False False False False  True  True
  True False False  True  True False False False False False False  True
 False False  True False False  True False False False False  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True]


In [20]:
X_train_backward = X_train[['Ship Mode_First Class','Ship Mode_Same Day', 'Ship Mode_Second Class', 'Ship Mode_Standard Class',
                            'State_Arkansas', 'State_Illinois', 'State_Indiana', 'State_Kansas', 'State_Maryland',
                            'State_Missouri', 'State_Nevada', 'State_North Carolina', 'State_Pennsylvania', 'State_Texas',
                            'State_Utah', 'State_Vermont', 'State_West Virginia', 'State_Wisconsin', 'Region_Central', 'Region_East', 'Region_South', 
                            'Region_West', 'Category_Furniture', 'Category_Office Supplies', 'Category_Technology',
                            'Sub-Category_Accessories', 'Sub-Category_Appliances', 'Sub-Category_Art', 'Sub-Category_Binders',
                            'Sub-Category_Bookcases', 'Sub-Category_Chairs', 'Sub-Category_Copiers', 'Sub-Category_Envelopes',
                            'Sub-Category_Fasteners', 'Sub-Category_Furnishings', 'Sub-Category_Labels',
                            'Sub-Category_Machines', 'Sub-Category_Paper', 'Sub-Category_Phones', 'Sub-Category_Storage',
                            'Sub-Category_Supplies', 'Sub-Category_Tables']]

X_test_backward = X_test[['Ship Mode_First Class','Ship Mode_Same Day', 'Ship Mode_Second Class', 'Ship Mode_Standard Class',
                            'State_Arkansas', 'State_Illinois', 'State_Indiana', 'State_Kansas', 'State_Maryland',
                            'State_Missouri', 'State_Nevada', 'State_North Carolina', 'State_Pennsylvania', 'State_Texas',
                            'State_Utah', 'State_Vermont', 'State_West Virginia', 'State_Wisconsin', 'Region_Central', 'Region_East', 'Region_South', 
                            'Region_West', 'Category_Furniture', 'Category_Office Supplies', 'Category_Technology',
                            'Sub-Category_Accessories', 'Sub-Category_Appliances', 'Sub-Category_Art', 'Sub-Category_Binders',
                            'Sub-Category_Bookcases', 'Sub-Category_Chairs', 'Sub-Category_Copiers', 'Sub-Category_Envelopes',
                            'Sub-Category_Fasteners', 'Sub-Category_Furnishings', 'Sub-Category_Labels',
                            'Sub-Category_Machines', 'Sub-Category_Paper', 'Sub-Category_Phones', 'Sub-Category_Storage',
                            'Sub-Category_Supplies', 'Sub-Category_Tables']]

model.fit(X_train_backward, y_train)

y_preds = model.predict(X_test_backward)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.000,0.000,0.000
2,0.000,0.000,0.000
3,0.845,0.845,1.079
4,1.000,1.000,1.276
5,1.000,1.000,1.276
6,1.000,1.000,1.276
7,1.000,1.000,1.276
8,1.000,1.000,1.276
9,1.000,1.000,1.276


In [21]:
PLS_model = PLSRegression()

PLS_model.fit(X_train, y_train)

y_preds = PLS_model.predict(X_test)

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})

values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

lift_table

,avg_predicted,avg_ordered,lift
decile,,,
1,0.077,0.000,0.000
2,0.232,0.009,0.012
3,0.619,0.833,1.063
4,0.778,1.000,1.276
5,0.834,1.000,1.276
6,0.885,1.000,1.276
7,0.939,1.000,1.276
8,1.010,1.000,1.276
9,1.111,1.000,1.276


## Conclusion

Looking at the results, I found that the first dataset had some increase in performance when I performed backward selection on the features which was surprising, but the partial least squares model performed the worst out of any model I have tested before. 

Dataset 2 had similar performance as it did with the previous regression models with the lift fluctuating per decile instead of having an increase. The partial least squares regression model was the best performing and started showing a true increase by decile. This makes sense since a lot of the features in the dataset are highly correlated due to being very similar categories.

Dataset 3 had effectively the same performance for all the methods tested in this notebook, and the same perforamnce as all the other models that have been tested.